# 🚀 EdgeGen AI — Live SD-Turbo GPU Inference Server (Google Colab)
**Problem Statement:** P3 — Resource-Efficient Text-to-Image Generation  
**Core Model:** `stabilityai/sd-turbo` (Few-Step Distilled Diffusion, FP16, VAE Slicing)  
**Classical CV:** Lanczos Resizing, Gray-World White Balance, CLAHE, Bilateral Filter  

---
### ⚡ Quick Start Checklist:
1. **Enable GPU:** Go to `Runtime` > `Change runtime type` > Select **T4 GPU** > `Save`.
2. **Run Cells Below:** Step through each cell or click `Runtime` > `Run all`.
3. **Enter ngrok Token:** Enter your free auth token when prompted (from [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)).
4. **Copy Public URL:** Copy the `https://xxxx.ngrok-free.app` URL and paste it into **EdgeGen AI > GPU Server > Connect Remote GPU**.

In [ ]:
# @title Step 1: Install Dependencies
!pip install -q fastapi "uvicorn[standard]" diffusers transformers accelerate safetensors opencv-python-headless pillow pyngrok
print("✅ Dependencies successfully installed!")

In [ ]:
# @title Step 2: Verify GPU & Load SD-Turbo Distilled Model in FP16
import os, sys, time, io, base64, uuid, threading, getpass
from typing import Optional, Dict, Any
import torch
from PIL import Image
import numpy as np
import cv2
from diffusers import AutoPipelineForText2Image
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import uvicorn
from pyngrok import ngrok

# Check GPU Availability
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected! Please go to Runtime > Change runtime type > T4 GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"🎮 Detected GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

print("📦 Loading stabilityai/sd-turbo in FP16...")
device = "cuda"
dtype = torch.float16

pipeline = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=dtype,
    variant="fp16",
    safety_checker=None
).to(device)

# Optimization: VAE & Attention Slicing for reduced memory footprint
pipeline.enable_vae_slicing()
pipeline.enable_attention_slicing(1)
print("✨ SD-Turbo loaded and optimized!")

In [ ]:
# @title Step 3: Classical Computer Vision Post-Processing Engine
class ClassicalCVEnhancer:
    def __init__(self):
        self.clahe = cv2.createCLAHE(clipLimit=1.8, tileGridSize=(8, 8))

    def enhance(self, image: Image.Image, aspect_ratio: str = "1:1") -> tuple[Image.Image, dict]:
        targets = {"1:1": (512, 512), "3:4": (480, 640), "4:3": (640, 480), "16:9": (704, 396)}
        tw, th = targets.get(aspect_ratio, (512, 512))
        
        # 1. Geometry-preserving Lanczos crop & resize
        ow, oh = image.size
        tr, or_ratio = tw / th, ow / oh
        if or_ratio > tr:
            nw = int(oh * tr)
            cropped = image.crop(((ow - nw) // 2, 0, (ow - nw) // 2 + nw, oh))
        else:
            nh = int(ow / tr)
            cropped = image.crop((0, (oh - nh) // 2, ow, (oh - nh) // 2 + nh))
        adapted = cropped.resize((tw, th), Image.Resampling.LANCZOS)

        # 2. Convert to OpenCV
        bgr = cv2.cvtColor(np.array(adapted), cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        blur_pre = float(cv2.Laplacian(gray, cv2.CV_64F).var())

        # 3. Gray-World White Balance
        b, g, r = cv2.split(bgr.astype(np.float32))
        avg_gray = (np.mean(b) + np.mean(g) + np.mean(r)) / 3.0
        if np.mean(b) > 0 and np.mean(g) > 0 and np.mean(r) > 0:
            b = np.clip(b * (0.5 + 0.5 * (avg_gray / np.mean(b))), 0, 255)
            g = np.clip(g * (0.5 + 0.5 * (avg_gray / np.mean(g))), 0, 255)
            r = np.clip(r * (0.5 + 0.5 * (avg_gray / np.mean(r))), 0, 255)
        balanced = cv2.merge((b, g, r)).astype(np.uint8)

        # 4. CLAHE on Luminance (LAB)
        lab = cv2.cvtColor(balanced, cv2.COLOR_BGR2LAB)
        l, a_ch, b_ch = cv2.split(lab)
        enhanced_l = self.clahe.apply(l)
        contrast_bgr = cv2.cvtColor(cv2.merge((enhanced_l, a_ch, b_ch)), cv2.COLOR_LAB2BGR)

        # 5. Bilateral edge-preserving denoising
        smoothed = cv2.bilateralFilter(contrast_bgr, d=5, sigmaColor=25, sigmaSpace=25)
        if blur_pre < 400:
            gaussian = cv2.GaussianBlur(smoothed, (0, 0), sigmaX=2.0)
            smoothed = cv2.addWeighted(smoothed, 1.25, gaussian, -0.25, 0)

        final_bgr = np.clip(smoothed, 0, 255).astype(np.uint8)
        final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB)
        
        # Metrics
        gray_post = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2GRAY)
        metrics = {
            "blur_laplacian_variance": round(float(cv2.Laplacian(gray_post, cv2.CV_64F).var()), 2),
            "is_blurry": False,
            "mean_brightness": round(float(np.mean(gray_post)), 2),
            "contrast_std": round(float(np.std(gray_post)), 2),
            "color_clipping_pct": round(float(np.sum((final_bgr <= 2) | (final_bgr >= 253)) / final_bgr.size * 100.0), 2)
        }
        return Image.fromarray(final_rgb), metrics

cv_enhancer = ClassicalCVEnhancer()
print("✅ Classical CV pipeline configured!")

In [ ]:
# @title Step 4: Setup FastAPI Server & Endpoints
app = FastAPI(title="EdgeGen AI Live Colab Engine")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

jobs_db: Dict[str, Dict[str, Any]] = {}

class GenerationRequest(BaseModel):
    prompt: str = Field(..., min_length=1)
    aspect_ratio: str = Field(default="1:1")
    style: str = Field(default="auto")
    negative_prompt: Optional[str] = None
    seed: Optional[int] = None

@app.get("/api/health")
async def health():
    return {"status": "healthy", "service": "EdgeGen AI Diffusion Backend", "engine_ready": True, "timestamp": time.time()}

@app.get("/api/models/status")
async def model_status():
    vram_mb = int(torch.cuda.memory_allocated(0) / (1024 * 1024))
    return {
        "online": True,
        "engine": "SD-Turbo Distilled Diffusion",
        "model": "stabilityai/sd-turbo",
        "device": "cuda",
        "precision": "fp16",
        "vram_allocated_mb": vram_mb,
        "memory_saving_enabled": True,
        "cv_pipeline_ready": True,
        "active_jobs": 0,
        "queue_length": 0,
        "api_version": "1.0.0",
        "hardware_summary": f"{gpu_name} (FP16, VRAM: {vram_mb}MB)"
    }

def execute_inference(prompt: str, aspect_ratio: str, style: str, seed: Optional[int] = None):
    enhanced = prompt
    styles = {
        "photorealistic": ", photorealistic photograph, 35mm lens, natural lighting, sharp focus, 8k",
        "cinematic": ", cinematic film still, dramatic volumetric lighting, Panavision 70mm, depth of field",
        "anime": ", anime style, vibrant aesthetic, crisp lineart, Makoto Shinkai aesthetic",
        "digital_art": ", detailed digital art, ArtStation trending, concept art, vivid rendering",
        "illustration": ", refined editorial illustration, clean shapes, harmonious color palette"
    }
    if style in styles:
        enhanced += styles[style]

    generator = torch.Generator(device="cuda").manual_seed(seed) if seed is not None else None
    torch.cuda.empty_cache()
    start = time.time()
    
    with torch.inference_mode():
        out = pipeline(prompt=enhanced, num_inference_steps=1, guidance_scale=0.0, width=512, height=512, generator=generator)
    raw = out.images[0]
    
    # Apply Classical CV
    final_img, metrics = cv_enhancer.enhance(raw, aspect_ratio)
    buf = io.BytesIO()
    final_img.save(buf, format="PNG")
    data_url = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode('utf-8')}"
    latency = round(time.time() - start, 2)
    return data_url, latency, metrics, final_img.size

@app.post("/api/generate")
async def generate(req: GenerationRequest):
    job_id = str(uuid.uuid4())
    try:
        data_url, latency, cv_metrics, (w, h) = execute_inference(req.prompt, req.aspect_ratio, req.style, req.seed)
        res = {
            "job_id": job_id,
            "status": "completed",
            "progress": 100,
            "message": "Image generated successfully",
            "prompt": req.prompt,
            "aspect_ratio": req.aspect_ratio,
            "style": req.style,
            "image_url": data_url,
            "width": w,
            "height": h,
            "latency_sec": latency,
            "cv_applied": True,
            "cv_metrics": cv_metrics,
            "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "completed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        jobs_db[job_id] = res
        return res
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/api/generate/{job_id}")
async def get_job(job_id: str):
    if job_id not in jobs_db:
        raise HTTPException(status_code=404, detail="Job not found")
    return jobs_db[job_id]

print("✅ FastAPI router & endpoints prepared!")

In [ ]:
# @title Step 5: Start Server & Launch Public ngrok Tunnel
token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    pass

if not token:
    token = os.environ.get("NGROK_AUTHTOKEN")

if not token:
    print("🔑 Enter your free ngrok token (from https://dashboard.ngrok.com/get-started/your-authtoken):")
    token = getpass.getpass("ngrok authtoken: ").strip()

ngrok.set_auth_token(token)

# Run Uvicorn in background thread
server_thread = threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True)
server_thread.start()
time.sleep(2)

# Expose via ngrok
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url.replace("http://", "https://")

print("\n" + "="*70)
print("🚀 EDGEGEN AI GPU INFERENCE SERVER IS LIVE!")
print("="*70)
print(f"👉 COPY THIS PUBLIC URL:  {public_url}")
print("="*70)
print("📋 INSTRUCTIONS:")
print("1. Open your EdgeGen AI web app.")
print("2. Click 'GPU Server' in the navigation bar.")
print("3. Click 'Connect Remote GPU' tab.")
print(f"4. Paste: {public_url}")
print("5. Click 'Save & Test'. The status indicator will turn green!")
print("="*70)

In [ ]:
# @title Step 6: Test In-Colab Generation
from IPython.display import display, HTML
import base64

print("🧪 Generating test sample to confirm SD-Turbo & CV pipeline...")
test_data, lat, metrics, dims = execute_inference("A cute red panda in a bamboo forest at golden hour, sharp focus", "1:1", "photorealistic")
print(f"🎉 SUCCESS! Generated {dims[0]}x{dims[1]} image in {lat}s (Laplacian Blur Score: {metrics['blur_laplacian_variance']})")

# Display test image in notebook
img_tag = f'<img src="{test_data}" style="width: 380px; border-radius: 12px; box-shadow: 0 8px 24px rgba(0,0,0,0.3);"/>'
display(HTML(f"<div>{img_tag}<p style='font-family:sans-serif;font-size:12px;color:#555;'>Prompt: A cute red panda in a bamboo forest · Latency: {lat}s</p></div>"))

### ⚠️ Free Google Colab Limitations & Best Practices

- **12-Hour Runtime Cap:** Free Google Colab instances have a maximum continuous runtime of 12 hours before the virtual machine terminates.
- **Idle Timeout Disconnect:** If your browser tab is closed or idle without activity for ~15–30 minutes, Google may disconnect the instance to reclaim resources.
- **GPU Compute Quota:** Google allocates free T4 GPUs based on dynamic availability. If you exceed the daily usage quota, wait 12–24 hours or use Google Colab Pro.
- **Dynamic ngrok URLs:** Free ngrok URLs change every time you restart `ngrok.connect()`. Whenever you restart the notebook, copy the newly printed URL into EdgeGen AI.